<a href="https://colab.research.google.com/github/Croop-weed/DeepLearning-parctice-/blob/main/data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf langchain-community langchain-text-splitters pdfplumber langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing install

In [2]:
import os
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LCDocument
import pdfplumber
from pypdf import PdfReader
import pandas as pd


/tmp/ipykernel_513/2612690753.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
path = "/content/SmartBin_AI_Project_Spec.pdf"

In [7]:
def extract_pdf_metadata(pdf_path):

    with open(pdf_path, 'rb') as f:
        reader = PdfReader(f)
        info = reader.metadata
        page_count = len(reader.pages)

    return {
        "title": info.title if info.title else os.path.basename(pdf_path),
        "author": info.author if info.author else "Unknown",
        "creator": info.creator if info.creator else "Unknown",
        "page_count": page_count
    }

In [8]:
print(extract_pdf_metadata(path))

{'title': 'SmartBin_AI_Project_Spec', 'author': 'Unknown', 'creator': 'Unknown', 'page_count': 10}


In [9]:
def extract_content_and_tables(pdf_path):
    structured_pages = []
    full_text_list = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()
            text = page.extract_text(layout=False)

            page_text = text if text else ""
            full_text_list.append(page_text)

            structured_pages.append({
                "page_number": page_num,
                "text": page_text,
                "tables": tables if tables else []
            })

    entire_document_text = "\n--- PAGE BREAK ---\n".join(full_text_list)
    return entire_document_text, structured_pages

In [10]:
def decision_mind_pdf_loader(pdf_path, chunk_size=1000, chunk_overlap=150):

    metadata = extract_pdf_metadata(pdf_path)
    full_text, structured_pages = extract_content_and_tables(pdf_path)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    parent_doc = LCDocument(
        page_content=full_text,
        metadata={
            "source": os.path.basename(pdf_path),
            "title": metadata["title"]
        }
    )

    langchain_chunks = text_splitter.split_documents([parent_doc])

    return {
        "metadata": metadata,
        "full_text": full_text,
        "structured_pages": structured_pages,
        "vector_chunks": langchain_chunks
    }

In [11]:
pdf_file_path = path

# Create fake entity keys to mock the database setup
mock_decision_id = uuid4()
mock_user_id = uuid4()

# Run your new unified pipeline
pipeline_output = decision_mind_pdf_loader(pdf_file_path)

# Build payload for your SQLAlchemy 'Document' model instance
document_db_record = {
    "id": uuid4(),                                      # Document PK
    "decision_id": mock_decision_id,                    # FK to your Decision model
    "uploaded_by": mock_user_id,                        # FK to your User model
    "filename": os.path.basename(pdf_file_path),
    "stored_filename": f"{uuid4()}.pdf",
    "mime_type": "application/pdf",
    "file_path": pdf_file_path,
    "file_size": os.path.getsize(pdf_file_path) if os.path.exists(pdf_file_path) else 0,
    "document_type": "PDF",                             # Maps to your DocumentType Enum
    "extracted_text": pipeline_output["full_text"]       # Text stored securely in Postgres
}

print("✅ Text successfully loaded and split via LangChain!")
print(f"🔹 Total Characters Saved for DB: {len(pipeline_output['full_text'])}")
print(f"🔹 Number of LangChain Chunks prepared for Vector Store: {len(pipeline_output['vector_chunks'])}")
print("\n--- LangChain Chunk 1 Sample ---")
print(document_db_record)
print(pipeline_output['vector_chunks'][0].page_content[:300] + "...")

✅ Text successfully loaded and split via LangChain!
🔹 Total Characters Saved for DB: 18512
🔹 Number of LangChain Chunks prepared for Vector Store: 22

--- LangChain Chunk 1 Sample ---
{'id': UUID('3c97b7b7-767f-4d07-b797-7f765dc33a2b'), 'decision_id': UUID('7087ff1c-64a2-4147-8211-d657b3269a36'), 'uploaded_by': UUID('3585a339-c9ed-47ff-9ffc-2606496298a4'), 'filename': 'SmartBin_AI_Project_Spec.pdf', 'stored_filename': '8c6f679b-663c-4294-9267-efaecb64ca46.pdf', 'mime_type': 'application/pdf', 'file_path': '/content/SmartBin_AI_Project_Spec.pdf', 'file_size': 211846, 'document_type': 'PDF', 'extracted_text': 'SmartBin AIProject Specification & Roadmap\nSmartBin AI\nDeep Learning-Based Garbage Overflow & Spillage Detection System\nProject Specification, Architecture & Build Roadmap\nPrepared as a self-build reference document\nJune 2026\nPage 1\n--- PAGE BREAK ---\nSmartBin AIProject Specification & Roadmap\nTable of Contents\nPage 2\n--- PAGE BREAK ---\nSmartBin AIProject Specification 

In [12]:
!pip install -q langchain-google-genai pydantic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 3.1 MB/s eta 0:00:00


In [13]:
import os
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# 1. Setup your Gemini API Key securely in Colab
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# 2. Define the Complete Unified Schema matching both decision.py and decision_analysis.py
class CompleteDecisionPayload(BaseModel):
    # Core Decision Fields
    title: str = Field(description="A concise title summarizing the decision.")
    topic: str = Field(description="1-2 words keyword classifying the topic (e.g., Infrastructure, Frontend, Security).")
    department: str = Field(description="The primary department impacted (e.g., Engineering, DevOps, HR, Finance).")
    problem_statement: str = Field(description="The underlying pain point, issue, or reason this decision process was triggered.")
    decision: str = Field(description="The final action item or choice that was officially approved.")
    reason: str = Field(description="The analytical justification for why this specific path was chosen over alternatives.")

    # Deep Analysis Fields (Matches decision_analyses table)
    summary: str = Field(description="An executive narrative summary of the entire decision context and analysis.")
    pros: list[str] = Field(description="A list of specific advantages and upside points resulting from this choice.")
    cons: list[str] = Field(description="A list of specific disadvantages, costs, or downsides resulting from this choice.")
    risks: list[str] = Field(description="Potential future failures, challenges, or vulnerabilities introduced by this decision.")
    alternatives: list[str] = Field(description="Other options that were actively considered but ultimately rejected.")
    assumptions: list[str] = Field(description="Core beliefs or unverified facts the team relied on when making this choice (e.g., 'Assuming the license fees remain flat').")
    confidence: float = Field(description="Your assessment score between 0.0 (low clarity) and 1.0 (highly documented/clear) on how confident we can be in this data based strictly on the text.")

# 3. The String Prompt Variable you requested for testing
EXTRACTION_PROMPT = """You are the Lead Knowledge Engineer for DecisionMind. Your objective is to transform un-structured corporate documents (PDF transcripts, proposals, meeting notes) into a dense, highly structured "Decision Intelligence Summary".

This summary will serve two purposes:
1. Human Comprehension: A clean, skimmable overview for company staff looking up past documents.
2. Machine Analysis: A structured input payload that an AI reasoning engine will parse to extract database fields (pros, cons, risks, options, assumptions).

---
INSTRUCTIONS:
1. Do not use conversational filler (e.g., "Here is a summary of..."). Start directly with the structured output.
2. Maintain strict fidelity to facts. Do not extrapolate, guess, or invent reasons not present in the text.
3. Preserve specific technical terms, metrics, cost estimations, vendor names, dates, and candidate options.
4. If a section is not mentioned in the source document, explicitly state "Not specified in document".

---
SOURCE DOCUMENT TEXT:
<document_text>
{document_text}
</document_text>

---
OUTPUT FORMAT:

# Executive Document Summary

##Document Overview
* **Title / Topic:** [Brief title or core topic of this document]
* **Primary Department / Stakeholders:** [e.g., Engineering, HR, Finance, Executive Board]
* **Core Subject:** [1-2 sentences explaining what problem or proposal this document addresses]

## The Core Problem & Context
[Summarize the underlying issue, market trigger, or operational challenge that forced this discussion. Include key constraints or deadlines mentioned.]

## Options & Trade-Offs Discussed
List all solutions or approaches evaluated in the text:
* **Option 1: [Name/Concept]**
  - Key Details: [Brief description]
  - Noted Advantages: [Stated benefits]
  - Noted Disadvantages/Costs: [Stated downsides]
* **Option 2: [Name/Concept]**
  - Key Details: [Brief description]
  - Noted Advantages: [Stated benefits]
  - Noted Disadvantages/Costs: [Stated downsides]

## Final Decision & Rationales
* **Selected Path:** [Explicitly state which option was chosen, or "Under Review" if undecided]
* **Primary Justification:** [Why this path was chosen over alternatives]
* **Status:** [e.g., APPROVED, REJECTED, PROPOSED, IN_PROGRESS]

## Stated Risks, Assumptions & Dependencies
* **Stated Risks:** [List any explicitly mentioned risks, failure modes, or potential security/financial threats]
* **Core Assumptions:** [List unverified assumptions the team made, e.g., "Assumes current API pricing holds for 2 years"]
* **Dependencies:** [Prerequisites needed before execution].
"""

In [14]:
# Initialize the Gemini Model via LangChain
# Gemini works exceptionally well with structured outputs out-of-the-box
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    max_retries=2
)

# Bind the unified Pydantic schema to the model
structured_gemini = llm.with_structured_output(CompleteDecisionPayload)

# Format our prompt string variable with your PDF text snippet
# (Assuming pipeline_output['full_text_for_db'] holds your extracted PDF string)
formatted_prompt = EXTRACTION_PROMPT.format(document_text=pipeline_output['full_text'])

print("🧠 Sending text payload to Gemini for structural synthesis...")
extracted_intelligence = structured_gemini.invoke(formatted_prompt)

# Print out your test properties to verify it worked!
print(extracted_intelligence.dict())

🧠 Sending text payload to Gemini for structural synthesis...
{'title': 'SmartBin AI: Edge-Deployed Two-Model Garbage Monitoring System', 'topic': 'Computer Vision', 'department': 'Engineering', 'problem_statement': 'Municipal solid waste collection runs on fixed schedules, leading to bins overflowing and garbage spilling onto surrounding areas before scheduled pickups occur.', 'decision': 'Implement a two-model computer vision pipeline deploying MobileNetV2 for fill-level classification and YOLOv8-nano for spillage detection on edge hardware (Raspberry Pi/Jetson Nano), integrated with a FastAPI backend and a weighted priority scoring algorithm.', 'reason': 'Splitting the tasks avoids over-engineering the fill-level task while preserving spatial localization for spillage. MobileNetV2 and YOLOv8-nano minimize computational overhead on edge hardware through depthwise separable convolutions and single-stage detection, respectively.', 'summary': 'The SmartBin AI project defines a deep learn

/tmp/ipykernel_513/1696319462.py:20: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(extracted_intelligence.dict())


In [15]:
for key,value in extracted_intelligence:
  print(key)
  print()
  print(value)
  print()

title

SmartBin AI: Edge-Deployed Two-Model Garbage Monitoring System

topic

Computer Vision

department

Engineering

problem_statement

Municipal solid waste collection runs on fixed schedules, leading to bins overflowing and garbage spilling onto surrounding areas before scheduled pickups occur.

decision

Implement a two-model computer vision pipeline deploying MobileNetV2 for fill-level classification and YOLOv8-nano for spillage detection on edge hardware (Raspberry Pi/Jetson Nano), integrated with a FastAPI backend and a weighted priority scoring algorithm.

reason

Splitting the tasks avoids over-engineering the fill-level task while preserving spatial localization for spillage. MobileNetV2 and YOLOv8-nano minimize computational overhead on edge hardware through depthwise separable convolutions and single-stage detection, respectively.

summary

The SmartBin AI project defines a deep learning-based system to optimize municipal waste collection. By deploying a two-model compute